# Comparing models 
## Introduction


## Model comparison: Regression vs. the mean
### The horizontal baseline


In [ ]:
import pandas as pd

data_borkman = pd.DataFrame({
    'per_C2022_fatacids': [ # percentage of C20-22 fatty acids
        17.9, 18.3, 18.3, 18.4, 18.4, 20.2, 20.3, 21.8, 21.9, 
        22.1, 23.1, 24.2, 24.4],
    'insulin_sensitivity': [ # insulin sensitivity index
        250, 220, 145, 115, 230, 200, 330, 400, 370, 260, 270, 530, 375]})

print("First row of the dataset:")
print(data_borkman.head())

In [ ]:
import numpy as np

# Separate X from y in pd.Series
X_borkman = data_borkman['per_C2022_fatacids']
y_borkman = data_borkman['insulin_sensitivity']

# Calculate the mean of Y
y_borkman_mean = y_borkman.mean()

# Fit the model using numpy.polyfit
coefficients_borkman = np.polyfit(X_borkman, y_borkman, deg=1)
slope_borkman, intercept_borkman = coefficients_borkman
print("Analysis of the Borkman data:")
print(f"Mean of insulin sensitivity (horizontal line) = \
{y_borkman_mean:.3f} mg/m²/min")
print("Results from linear regression (NumPy):")
print(" Intercept =", round(intercept_borkman, 3))
print(" Slope =", round(slope_borkman, 3))

In [ ]:
#| fig-subcap: 
#|   - "Linear regression line fitted to the data"
#|   - "Simple horizontal line (null hypothesis)"
#| layout-ncol: 2

import matplotlib.pyplot as plt

# Plot 1: linear regression model
plt.figure(figsize=(3.5, 3))
plt.plot(X_borkman, y_borkman, 'k+', ms=8)
plt.ylim(0, 600)

# Calculate and plot the regression line
y_borkman_line = slope_borkman * X_borkman + intercept_borkman
plt.plot(X_borkman, y_borkman_line, 'r-', lw=3)

plt.xlabel('%C20-22 fatty acids')
plt.ylabel('Insulin sensitivity (mg/m²/min)')
plt.title('Linear regression model')
plt.show()


# Plot 2: null hypothesis model (horizontal line)
plt.figure(figsize=(3.5, 3))
plt.plot(X_borkman, y_borkman, 'k+', ms=8)
plt.ylim(0, 600)

# Draw an horizontal line
plt.hlines(
    y=y_borkman_mean, xmin=X_borkman.min(), xmax=X_borkman.max(),
    colors='b', linestyles='-', lw=3)

plt.xlabel('%C20-22 fatty acids')
plt.ylabel('Insulin sensitivity (mg/m²/min)')
plt.title('Null hypothesis model (mean)')
plt.show()

### R-squared
### Sums of squares and variation


In [ ]:
# Define functions to compute RSS and estimate y
def compute_sse(y_estimate, y):
    return sum(np.power(y - y_estimate, 2))

def estimate_y(X, intercept, slope):
    return intercept + slope * X

In [ ]:
# Calculate RSS and TSS
sse_borkman = compute_sse(
    estimate_y(X=X_borkman, intercept=intercept_borkman, slope=slope_borkman),
    y_borkman)
sst_borkman = compute_sse(y_borkman_mean, y_borkman)
# sst_borkman = (y_borkman  - y_borkman.mean()).pow(2).sum()
ssr_borkman = sst_borkman - sse_borkman  # TSS - RSS

# Print RSS, TSS and R²
print("Goodness of fit metrics (Borkman data):")
print(
    f"Scatter around the horizontal line (TSS or SST) = {sst_borkman:.1f}")
print(
    f"Scatter around the regression line (RSS or SSE) = {sse_borkman:.1f}")
print(f"Variability explained by the regression model (SSR) = \
{ssr_borkman:.1f}")
print()

print(f"Scatter around the regression line (SSE/SST) accounts for \
{1 - (ssr_borkman/sst_borkman):2.1%}\n  of the total variation")
print(f"Proportion of variation explained by the regression line (SSR/SST) \
= {ssr_borkman/sst_borkman:2.1%}")
print(f"R² = {1 - sse_borkman/sst_borkman:.4f}")

### Analysis of variance
#### Sources of variation


#### Visualizing the residuals


In [ ]:

import seaborn as sns

# Residuals from linear regression
residuals_regression = (
    y_borkman - estimate_y(
        X_borkman, intercept_borkman, slope_borkman)).to_numpy()
# Residuals from horizontal line (mean)
residuals_mean = (y_borkman - y_borkman_mean).to_numpy()

plt.figure(figsize=(3.5, 3))

sns.stripplot(
    data=[residuals_regression, residuals_mean],
    color='k')
plt.axhline(
    y=0, color='orange', linestyle='--', linewidth=2,)
plt.ylabel("Vertical distance to prediction")
plt.xticks([0, 1], ['linear regression', 'null hypothesis'])
plt.title("Residuals from models");

#### Degrees of freedom


In [ ]:
# Number of y-observations
n_borkman = y_borkman.size

# Degrees of freedom:
df_regression_borkman = 1  # DF of regression/model (explained variation)
df_error_borkman = n_borkman - 2  # DF of residuals (unexplained variation)
df_total_borkman = df_regression_borkman + df_error_borkman

print("Degrees of freedom (Borkman data):")
print(f"Degrees of freedom regression = {df_regression_borkman}")
print(f"Degrees of freedom error = {df_error_borkman}")
print(f"Degrees of freedom total = {df_total_borkman}")

#### Mean squares


In [ ]:
# Calculate mean squares
mean_square_regression_borkman = ssr_borkman / df_regression_borkman
mean_square_error_borkman = sse_borkman / df_error_borkman

print("Mean squares (Borkman data):")
print(
    f"Mean square regression (MSR) = {mean_square_regression_borkman:.1f}")
print(f"Mean square error (MSE) = {mean_square_error_borkman:.1f}")

#### F-ratio


In [ ]:
# Calculate F-statistic
f_statistic_borkman = (
    mean_square_regression_borkman / mean_square_error_borkman)
print(f"F-ratio (Borkman data) = {f_statistic_borkman:.2f}")

#### P value


In [ ]:
from scipy.stats import f as f_dist

# Calculate the P value using the survival function of the F-distribution
p_value_borkman = f_dist(
    dfn=df_regression_borkman, dfd=df_error_borkman).sf(f_statistic_borkman)

print(f"P value of the F-test (Borkman data) = {p_value_borkman:.5f}")

#### Visualizing critical values


In [ ]:

# Significance level (alpha)
α = 0.05

# Calculate critical F-value
f_crit_borkman = f_dist(
    dfn=df_regression_borkman, dfd=df_error_borkman).ppf(1 - α)

# Generate x values for plotting
x_f = np.linspace(0, 16, 500)
hx_f = f_dist(df_regression_borkman, df_error_borkman).pdf(x_f)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x_f, hx_f, lw=2, color='black')

# Critical value
plt.axvline(
    x=f_crit_borkman,
    color='orangered', linestyle='--',
    label=f"F* ({f_crit_borkman:.2f})")

# Alpha area
plt.fill_between(
    x_f[x_f >= f_crit_borkman],
    hx_f[x_f >= f_crit_borkman],
    color='tomato',
    label=f"α ({α})")

# F-statistic
plt.axvline(
    x=f_statistic_borkman,
    color='limegreen', linestyle='-.',
    label=f"F ({f_statistic_borkman:.2f})")

# P-value area
plt.fill_between(
    x_f[x_f >= f_statistic_borkman],
    hx_f[x_f >= f_statistic_borkman],
    color='greenyellow',
    label=f"P ({p_value_borkman:.3f})")

plt.xlabel("F")
plt.ylabel('Density')
plt.ylim(0, .2)
plt.title(
    f"F-distribution (DFn={df_regression_borkman}, DFd={df_error_borkman})")
plt.margins(x=0.02, y=0)
plt.legend();

#### ANOVA table


#### Generating the ANOVA table in Python


In [ ]:
import statsmodels.formula.api as smf
import warnings

# Suppress all UserWarnings, incl. messages related to small sample size
warnings.filterwarnings("ignore", category=UserWarning)

model_linear_borkman = smf.ols(
    "insulin_sensitivity ~ per_C2022_fatacids", data=data_borkman)
results_linear_borkman = model_linear_borkman.fit()

# Classical output of the OLS result table
print("Assessment table extracted from statsmodels OLS results \
(Borkman data):")
print(results_linear_borkman.summary().tables[0])

In [ ]:
from statsmodels.stats.anova import anova_lm

print("ANOVA table generated with statsmodels (Borkman data):")
print(anova_lm(results_linear_borkman, type=3).round(4))

### Nested models


## Unpaired t-test as a special case of regression
### Two-group comparison
### Dummy variables


### Regression analysis of group differences


In [ ]:
# Data from the chapter on unpaired t-test
old = np.array([20.8, 2.8, 50, 33.3, 29.4, 38.9, 29.4, 52.6, 14.3])
yng = np.array([45.5, 55, 60.7, 61.5, 61.1, 65.5, 42.9, 37.5])

# Create the DataFrame with the dummy variable 'X'
data_frazier = pd.DataFrame({
    'X': [0] * len(old) + [1] * len(yng),  # 0 for old, 1 for young rats
    'y': np.concatenate([old, yng])        # Combine the arrays
})

# Print a random sample of the data
print("Random sample of the Frazier dummified dataset:")
print(data_frazier.sample(5))

In [ ]:
#| fig-subcap: 
#|   - "Data distribution within each group"
#|   - "Linear regression plot"
#| layout-ncol: 2

# Box and swarm plot
plt.figure(figsize=(3.5, 3))

sns.boxplot(
    x='X', y='y', data=data_frazier,
    width=0.5, color='black',
    boxprops={'facecolor': 'none'},  # Make the boxplot transparent
)
sns.swarmplot(
    x='X', y='y', data=data_frazier,
    color='black',)

plt.ylabel(r"$\%\text{E}_\text{max}$")
plt.xticks([0, 1], ['Old', 'Young'])
plt.xlabel("Group")
plt.ylim(0, 80)
plt.title("Box and swarm plots")
plt.show()

# Regression plot
plt.figure(figsize=(3.5, 3))

sns.regplot(
    x='X', y='y', data=data_frazier,
    ci=False, color='k',
    line_kws={'color': 'red'})
plt.xticks([0, 1])
plt.ylim(0, 80)
plt.ylabel(r"$\%\text{E}_\text{max}$")
plt.title("Linear regression")
plt.show()

#### Performing regression with dummy variables


In [ ]:
# Fit the linear regression model using the formula API
model_ttest_frazier = smf.ols('y ~ X', data=data_frazier)
results_ttest_frazier = model_ttest_frazier.fit()

print("statsmodels OLS results (Frazier data):")
print(results_ttest_frazier.summary2())

In [ ]:
print("Group summary statistics (Frazier data):")
print(data_frazier.groupby('X').describe().round(4))
print()
print(f"Mean difference between groups = \
{round(data_frazier.groupby('X').mean().apply(np.diff).values[0][0], 4)}")

In [ ]:
import pingouin as pg

# Student's t-test using Pingouin
results_ttest_pingouin_frazier = pg.ttest(yng, old, correction=False)
print("Student's t-test results from Pingouin (Frazier data):")
print(results_ttest_pingouin_frazier.round(4))

#### Goodness of fit


In [ ]:
# Get the predicted values from the statsmodels model
y_frazier = data_frazier['y']
y_pred_frazier = results_ttest_frazier.predict()

# Calculate RSS and TSS
sse_frazier = compute_sse(y_pred_frazier, y_frazier)
sst_frazier = compute_sse(y_frazier.mean(), y_frazier)  # SST

ssb_frazier = sst_frazier - sse_frazier  # TSS - RSS

print("Goodness of fit metrics (Frazier data):")
print(f"Total scatter (SST) = {sst_frazier:.1f}")
print(f"Scatter error (SSE) = {sse_frazier:.1f}")
print(f"Scatter between groups (SSB) = {ssb_frazier:.1f}")
print()
print(f"Proportion of scatter error (SSE/SST): \
{1 - (ssb_frazier/sst_frazier):2.1%} of the total scatter")
print(f"Proportion of scatter between groups (SSB/SST) = \
{ssb_frazier/sst_frazier:2.1%}")
print(f"R² = {1 - sse_frazier/sst_frazier:.4f}")

#### ANOVA table for t-test


In [ ]:
n_frazier = y_frazier.size

# Degrees of freedom
df_between_frazier = 1  # Number of groups minus 1
df_error_frazier = n_frazier - 2  # We estimate 2 group means

f_statistic_frazier = (
    ((sst_frazier - sse_frazier) / df_between_frazier)
    /
    (sse_frazier / df_error_frazier)
)

# Calculate the P value using the survival function of the F-distribution
p_value_frazier = f_dist(
    dfn=df_between_frazier, dfd=df_error_frazier).sf(f_statistic_frazier)

print("F-test (Frazier data):")
print("-" * 35)
print(f"Degrees of freedom between groups = {df_between_frazier}")
print(f"Degrees of freedom error = {df_error_frazier}")
print(f"Degrees of freedom total = {df_between_frazier + df_error_frazier}")
print()
print(f"Mean square between groups = {ssb_frazier/df_between_frazier:.1f}")
print(f"Mean square error (MSE) = {sse_frazier/df_error_frazier:.1f}")
print()
print(f"F-ratio = {f_statistic_frazier:.3f}")
print(f"P value = {p_value_frazier:.4f}")

In [ ]:

# Calculate critical F-value
f_crit_frazier = f_dist(
    dfn=df_between_frazier, dfd=df_error_frazier).ppf(1 - α)

# Generate x values for plotting
x_f = np.linspace(0, 14, 500)
hx_f = f_dist.pdf(x_f, df_between_frazier, df_error_frazier)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x_f, hx_f, lw=2, color='black')

# Critical value
plt.axvline(
    x=f_crit_frazier,
    color='orangered', linestyle='--',
    label=f"F* ({f_crit_frazier:.2f})")

# Alpha area
plt.fill_between(
    x_f[x_f >= f_crit_frazier],
    hx_f[x_f >= f_crit_frazier],
    color='tomato',
    label=f"α ({α})")

# F-statistic
plt.axvline(
    x=f_statistic_frazier,
    color='limegreen', linestyle='-.',
    label=f"F ({f_statistic_frazier:.2f})")

# P-value area
plt.fill_between(
    x_f[x_f >= f_statistic_frazier],
    hx_f[x_f >= f_statistic_frazier],
    color='greenyellow',
    label=f"P ({p_value_frazier:.3f})")

plt.xlabel("F")
plt.ylabel('Density')
plt.ylim(0, .2)
plt.title(f"F-distribution (DFn={df_between_frazier}, DFd={df_error_frazier})")
plt.margins(x=0.01, y=0)
plt.legend();

In [ ]:
print("ANOVA table generated with statsmodels (Frazier data):")
print(anova_lm(results_ttest_frazier, type=3).round(3))

### F-ratio and t-statistic


In [ ]:
# Extract the t-value for the dummy variable (X)
t_value_frazier = results_ttest_frazier.tvalues['X']
# t_value_frazier = results_ttest_pingouin_frazier.loc['T-test', 'T']

print(f"t-value from regression: {t_value_frazier:.4f}")
print(f"t-value squared: {t_value_frazier**2:.5f}")
print(f"F-value from ANOVA: {f_statistic_frazier:.5f}")

### Comparing `y~1` to `y~X`: The grand mean model


In [ ]:
# Fit the grand mean model
model_grandmean_frazier = smf.ols('y ~ 1', data=data_frazier)
results_grandmean_frazier = model_grandmean_frazier.fit()

print("Grand mean model statsmodels results (Frazier data):")
print(results_grandmean_frazier.summary2().tables[1])

In [ ]:
# Calculate the grand mean and its standard error
grand_mean_frazier = data_frazier['y'].mean()
se_grand_mean_frazier = data_frazier['y'].sem()

print("Grand mean and its standard error (Frazier data):")
print(f"Mean = {grand_mean_frazier:.3f}")
print(f"Standard error = {se_grand_mean_frazier:.4f}")

In [ ]:

_, axes = plt.subplots(
    nrows=1, ncols=2, figsize=(6, 3), sharey=True, sharex=True)

# Plot residuals for the grand mean model
axes[0].scatter(
    data_frazier['X'],
    results_grandmean_frazier.resid,
    color='grey',
    label='Grand mean')
axes[0].axhline(0, color='orange', linestyle='--')
axes[0].set_xlabel('X')
axes[0].set_xticks([0, 1])
axes[0].set_ylabel('Residuals')
axes[0].set_title('Total variation (SST)')
axes[0].legend()

# Plot residuals for the group mean model
ix = data_frazier['X'] == 0  # Mask for group selection
axes[1].scatter(
    data_frazier.loc[ix, 'X'],
    results_ttest_frazier.resid[ix],
    color='dodgerblue',
    label='Old group mean')
axes[1].scatter(
    data_frazier.loc[~ix, 'X'],
    results_ttest_frazier.resid[~ix],
    color='orangered',
    label='Young group mean')
axes[1].axhline(0, color='orange', linestyle='--')
axes[1].set_xlabel('X')
axes[1].set_xticks([0, 1])
axes[1].set_ylabel('Residuals')
axes[1].set_title("Within-group (SSW)")
axes[1].legend()
plt.tight_layout()
sns.despine();

In [ ]:
# Calculate the sums of squares
sst_grandmean = compute_sse(grand_mean_frazier, data_frazier['y'])
ssw_grandmean = sst_grandmean
ssb_grandmean = 0

print("Goodness of fit metrics (Frazier data; grand mean model):")
print(f"Total scatter (SST) = {sst_grandmean:.1f}")
print(f"Scatter within groups (SSW) = {ssw_grandmean:.1f}")
print(f"Scatter between groups (SSB): {ssb_grandmean:.1f}")
print()
print(f"Proportion of scatter within groups (SSW/SST): \
{ssw_grandmean/sst_grandmean:2.1%} of the total scatter")
print(f"Proportion of scatter between groups (SSB/SST) = \
{ssb_grandmean/sst_grandmean:2.1%}")
print(f"R² = {1 - ssw_grandmean/sst_grandmean:.4f}")

In [ ]:
# Get the sum of squared residuals (SSR) for each model
sst_grandmean = results_grandmean_frazier.ssr
ssw_grandmean = results_ttest_frazier.ssr
ssb_grandmean = sst_grandmean - ssw_grandmean

# Degrees of freedom
n_groups_grandmean = 2
df_between_grandmean = n_groups_grandmean - 1
df_within_grandmean = n_frazier - n_groups_grandmean

# Calculate the F-ratio
f_statistic_grandmean = (
    (ssb_grandmean / df_between_grandmean)
    /
    (ssw_grandmean / df_within_grandmean)
)

# Calculate the P value
p_value_grandmean = f_dist(
    dfn=df_between_grandmean,
    dfd=df_within_grandmean).sf(f_statistic_grandmean)
# Print all results
print("F-test grand mean vs. dummy variables:")
print("-"*35)
print(f"SST = {sst_grandmean:.1f}")
print(f"SSW = {ssw_grandmean:.1f}")
print(f"SSB = {ssb_grandmean:.1f}")
print()
print(f"Degrees of freedom between = {df_between_frazier}")
print(f"Degrees of freedom within = {df_error_frazier}")
print(f"Degrees of freedom total = {df_between_frazier + df_error_frazier}")
print()
print(f"Mean square between groups = {ssb_frazier/df_between_frazier:.1f}")
print(f"Mean square error (MSE) = {sse_frazier/df_error_frazier:.1f}")
print()
print(f"F ratio = {f_statistic_grandmean:.3f}")
print(f"P value = {p_value_grandmean:.4f}")

In [ ]:
print("ANOVA comparison: grand mean model vs. t-test recast model:")
print(anova_lm(results_grandmean_frazier, results_ttest_frazier))

## Conclusion
